# Set up in cloud

### For Colab notebooks, start here

In [ ]:
!git clone https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

In [ ]:
%cd grouping-trainer

### For Workbench notebooks, start here

In [ ]:
!git rev-parse --short HEAD

After running this `pip install` cell, restart the notebook session. TODO: activate venv instead

In [ ]:
!pip install -e .

In [ ]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

In [ ]:
!mkdir gte-finetuned
!gsutil -m cp -r gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training gte-finetuned/

In [ ]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

# Run

In [ ]:
import os
import json
from datetime import datetime
import time

import numpy as np
import polars as pl
from pydantic import BaseModel, field_serializer
from sentence_transformers.util import pairwise_cos_sim
import torch
from tqdm.auto import tqdm

import grouping_trainer as gt
import utils

In [ ]:
class ModelConfig(BaseModel):
    name: str
    path: str
    truncate_dim: int | None = None
    batch_size: int = 1
    model_kwargs: dict | None = None
    compile_kwargs: dict | None = None

    @field_serializer("model_kwargs")
    def serialize_model_kwargs(self, v: dict | None) -> dict:
        if v is None:
            return None
        return {k: str(val) if isinstance(val, torch.dtype) else val for k, val in v.items()}


class ModelConfigs(BaseModel):
    model_configs: list[ModelConfig]


class DataConfig(BaseModel):
    val_df_path: str
    sample_size: int | None = None

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

In [ ]:
RUN_SHORTNAME = "val-temp-delete"
DATA_CONFIG = DataConfig(
    val_df_path="final_csvs/val.csv",
    sample_size=100,
    # sample_size=None,
)
MODEL_CONFIGS = ModelConfigs(
    model_configs=[
        ModelConfig(
            name="prod",
            path="issue_grouping_v1/embeddings",
            truncate_dim=None,
        ),
        ModelConfig(
            name="gte-finetuned",
            path="gte-finetuned/training",
            truncate_dim=64,
            model_kwargs=dict(
                dtype=torch.bfloat16,
                attn_implementation="sdpa",
                # attn_implementation="flash_attention_2",  # hell
            ),
            compile_kwargs={"mode": "default"},  # reduce-overhead didn't empirically help
        ),
    ]
)
OUTPUT_DIR = f"./{timestamp}-{RUN_SHORTNAME}"

In [ ]:
def encode_timed(model: gt.utils.SentenceTransformer, texts: list[str]) -> tuple[np.ndarray, list[float]]:
    times: list[float] = []
    embeddings: list[np.ndarray] = []
    for text in texts:
        start = time.monotonic()
        emb = model.encode(text, convert_to_numpy=True)
        end = time.monotonic()
        times.append(end - start)
        embeddings.append(emb)
    return np.array(embeddings), times

In [ ]:
df = utils.load_val_df(path=DATA_CONFIG.val_df_path, sample_size=DATA_CONFIG.sample_size)
print(df.shape)
print(df.columns)

In [ ]:
for model_config in tqdm(MODEL_CONFIGS.model_configs, desc="Models"):
    print(model_config)

    model = gt.utils.SentenceTransformer(
        model_config.path,
        trust_remote_code=True,
        truncate_dim=model_config.truncate_dim,
        model_kwargs=model_config.model_kwargs,
    )
    if model_config.compile_kwargs:
        model.compile(**model_config.compile_kwargs)
    _ = model.encode("warm up")

    query_texts = df["query_stacktrace_string"].to_list()
    query_embeddings, query_times = encode_timed(model, query_texts)

    candidate_texts = df["candidate_stacktrace_string"].to_list()
    candidate_embeddings, candidate_times = encode_timed(model, candidate_texts)

    cos_sims = pairwise_cos_sim(query_embeddings, candidate_embeddings).detach().cpu().numpy()

    df = df.with_columns(
        [
            pl.Series(name=f"cos_sim_{model_config.name}", values=cos_sims),
            pl.Series(name=f"query_encode_time_{model_config.name}", values=query_times),
            pl.Series(name=f"candidate_encode_time_{model_config.name}", values=candidate_times),
        ]
    )

# Upload

In [ ]:
os.mkdir(OUTPUT_DIR)

In [ ]:
with open(f"{OUTPUT_DIR}/model_configs.json", "w") as f:
    json.dump(MODEL_CONFIGS.model_dump(), f, indent=4)

with open(f"{OUTPUT_DIR}/data_config.json", "w") as f:
    json.dump(DATA_CONFIG.model_dump(), f, indent=4)

In [ ]:
df.write_csv(f"{OUTPUT_DIR}/similarities.csv")

In [ ]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/similarities/{OUTPUT_DIR}

In [ ]:
!gsutil -m cp -r run.ipynb gs://grouping-data/similarities/{OUTPUT_DIR}